In [1]:
import requests
import json
from pyshacl import validate
from rdflib import Graph, URIRef
from typing import List, Dict

# --- CONFIGURATION (Ensure these URLs are correct for your environment) ---

# 1. Authoritative DCAT-US 3.0 SHACL Rules (The definitive v3.0 standard)
SHACL_RULES_URL = "https://raw.githubusercontent.com/DOI-DO/dcat-us/main/shacl/dcat-us_3.0_shacl_shapes.ttl"

# 2. Target DCAT-US 1.1 Data (Example: Minimal/Empty Catalog for Max Failures)
AGENCY_DATA_URL = "https://ngda-transportation-geoplatform.hub.arcgis.com/api/feed/dcat-us/1.1?id=23d91bd988ac4fc9943128965bddfa37"


def fetch_and_load_graph(url: str, format: str) -> Graph:
    """Fetches content from a URL and loads it into an RDF graph."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        graph = Graph()
        # The 'json-ld' format parser handles the conversion from JSON-LD to RDF triples
        graph.parse(data=response.text, format=format)
        return graph
    except requests.exceptions.RequestException as e:
        print(f"FATAL ERROR: Could not fetch {url}. Validation aborted: {e}")
        return None
    except Exception as e:
        print(f"FATAL ERROR: Could not parse content from {url} as {format}: {e}")
        return None


def generate_conversion_report(data_url: str, shacl_url: str):
    # 1. Load the SHACL Rules (Shapes Graph)
    print("1. Loading DCAT-US 3.0 SHACL Rules...")
    shacl_graph = fetch_and_load_graph(shacl_url, 'turtle')
    if shacl_graph is None: return
        
    # 2. Load the Agency's DCAT-US 1.1 Data (Data Graph)
    print(f"2. Loading Agency Data from {data_url}...")
    data_graph = fetch_and_load_graph(data_url, 'json-ld')
    if data_graph is None: return

    # 3. Perform SHACL Validation
    print("3. Running Comprehensive SHACL validation...")
    conforms, results_graph, results_text = validate(
        data_graph, 
        shacl_graph=shacl_graph,
        inference='rdfs', # Ensures RDFS rules are applied (e.g., subclass relationships)
        abort_on_error=False
    )

    # 4. Analyze the Validation Report to Create the Guide
    print("\n" + "="*75)
    print(f"| DCAT-US 1.1 to 3.0 Conversion Analysis: {'CONFORMANT (PASS)' if conforms else 'VIOLATIONS FOUND (CONVERSION REQUIRED)'}")
    print("="*75)

    if conforms:
        print("🎉 The catalog meets all DCAT-US 3.0 Mandatory requirements.")
        
    else:
        # Use SPARQL to query the machine-readable results_graph for *all* failures
        query = """
        SELECT ?focusNode ?severity ?message ?sourceShape
        WHERE {
            ?result a sh:ValidationResult .
            ?result sh:focusNode ?focusNode .
            ?result sh:resultSeverity ?severity .
            ?result sh:resultMessage ?message .
            ?result sh:sourceShape ?sourceShape .
        }
        """
        
        severity_map = {
            str(URIRef("http://www.w3.org/ns/shacl#Violation")): "🛑 MANDATORY",
            str(URIRef("http://www.w3.org/ns/shacl#Warning")): "⚠️ RECOMMENDED",
            str(URIRef("http://www.w3.org/ns/shacl#Info")): "ℹ️ OPTIONAL"
        }
        
        findings: List[Dict] = []
        for row in results_graph.query(query):
            # Clean up URIs for human readability
            entity = str(row.focusNode).split('#')[-1].split('/')[-1].replace('_',' ').title()
            severity = severity_map.get(str(row.severity), "UNKNOWN")
            
            # Extract the property from the message if possible (heuristic for clearer messages)
            message = str(row.message).split('on property')[0].strip()
            
            findings.append({
                'Severity': severity,
                'Entity': entity,
                'Violation': message,
                'Action': '' # Placeholder for the final action text
            })
            
        output_markdown_guide(findings)

def output_markdown_guide(findings: List[Dict]):
    """Generates the human-readable markdown conversion guide."""
    
    print("## 📄 DCAT-US 1.1 to 3.0 Conversion Guide\n")
    print("This guide translates the technical validation report into actionable tasks for your catalog.\n")

    # --- GLOSSARY (For non-experts) ---
    print("### 📚 Glossary of Key Terms\n")
    print("| Term | Definition |\n| :--- | :--- |")
    print("| **dcat:Catalog** | The top-level file (`data.json`) representing your entire data inventory. |")
    print("| **dcat:Dataset** | The core data asset (e.g., 'Annual Health Statistics'). |")
    print("| **Mandatory (M)** | **MUST** be fixed. An absolute failure to meet the v3.0 standard. |")
    print("| **Recommended (R)** | **SHOULD** be fixed. High-value modernization for improved discoverability (FAIR). |")
    print("| **SHACL** | The formal **rulebook** used to validate the semantic structure of your data. |")

    # --- MANDATORY GAPS SECTION ---
    print("\n---\n")
    print("### 🛑 Mandatory Field Gaps (Task 1: MUST FIX for v3.0 Compliance)\n")
    mandatory_findings = [f for f in findings if 'MANDATORY' in f['Severity']]
    
    if mandatory_findings:
        print("The following properties are required by v3.0 but are missing or invalid in your catalog. These must be addressed first.\n")
        print("| Status | Entity (Focus Node) | Missing Property / Structural Violation | Required Action (Conversion Step) |")
        print("| :--- | :--- | :--- | :--- |")
        
        for f in mandatory_findings:
            # Map the technical finding to a non-expert instruction
            if 'Less than 1 values on' in f['Violation']:
                 action = f"Add the missing {f['Violation'].split('->')[-1].strip()} property."
            elif 'does not conform to' in f['Violation']:
                 action = f"The value for this property has an INVALID structure. It must be a structured entity (e.g., vcard:Contact) with required sub-fields."
            else:
                 action = f"Correct the missing or invalid property value based on DCAT-US 3.0 rules. ({f['Violation']})"

            # Identify if the core entities are missing (most likely case for empty 1.1 file)
            if 'Catalog' in f['Entity'] and 'dcat:dataset' in f['Violation']:
                action = "The catalog contains ZERO Datasets. You MUST add at least one fully compliant dcat:Dataset entity."

            print(f"| {f['Severity']} | `{f['Entity']}` | {f['Violation']} | **{action}** |")
    else:
        print("✅ No Mandatory (M) DCAT-US 3.0 requirements were violated. Proceed to Recommended checks.")

    # --- RECOMMENDED ENHANCEMENTS SECTION ---
    print("\n---\n")
    print("### ⚠️ Recommended Enhancements (Task 2: High-Value Modernization)\n")
    recommended_findings = [f for f in findings if 'RECOMMENDED' in f['Severity']]
    
    if recommended_findings:
        print("These issues point to new features in DCAT-US 3.0 that significantly improve data discoverability, governance, and trust (FAIR principles).\n")
        print("| Priority | Entity (Focus Node) | Enhancement Area | Conversion Step |")
        print("| :--- | :--- | :--- | :--- |")
        
        for f in recommended_findings:
            if 'dcat-us:geographicBoundingBox' in f['Violation']:
                area = "Geospatial Unification"
                action = "Replace simple 'dct:spatial' text with the structured 'dcat-us:GeographicBoundingBox' class coordinates."
            elif 'checksum' in f['Violation']:
                area = "Data Integrity"
                action = "Add the 'spdx:checksum' to the Distribution, calculated using a standard algorithm (e.g., SHA-256)."
            elif 'controlled vocabulary' in f['Violation']:
                area = "Controlled Vocabulary"
                action = "Ensure the property value is a recognized URI from the official DCAT-US 3.0 Controlled Vocabulary list."
            else:
                area = "General Modernization"
                action = f"Consider adding the referenced property for richer context. ({f['Violation']})"
                
            print(f"| {f['Severity']} | `{f['Entity']}` | {area} | **{action}** |")
    else:
        print("✅ No major Recommended (R) issues found.")

    print("\n" + "="*75)


# --- FINAL EXECUTION ---
if __name__ == "__main__":
    generate_conversion_report(AGENCY_DATA_URL, SHACL_RULES_URL)

1. Loading DCAT-US 3.0 SHACL Rules...
2. Loading Agency Data from https://ngda-transportation-geoplatform.hub.arcgis.com/api/feed/dcat-us/1.1?id=23d91bd988ac4fc9943128965bddfa37...


Usage of abort_on_error is deprecated. Use abort_on_first instead.


3. Running Comprehensive SHACL validation...

| DCAT-US 1.1 to 3.0 Conversion Analysis: VIOLATIONS FOUND (CONVERSION REQUIRED)
## 📄 DCAT-US 1.1 to 3.0 Conversion Guide

This guide translates the technical validation report into actionable tasks for your catalog.

### 📚 Glossary of Key Terms

| Term | Definition |
| :--- | :--- |
| **dcat:Catalog** | The top-level file (`data.json`) representing your entire data inventory. |
| **dcat:Dataset** | The core data asset (e.g., 'Annual Health Statistics'). |
| **Mandatory (M)** | **MUST** be fixed. An absolute failure to meet the v3.0 standard. |
| **Recommended (R)** | **SHOULD** be fixed. High-value modernization for improved discoverability (FAIR). |
| **SHACL** | The formal **rulebook** used to validate the semantic structure of your data. |

---

### 🛑 Mandatory Field Gaps (Task 1: MUST FIX for v3.0 Compliance)

The following properties are required by v3.0 but are missing or invalid in your catalog. These must be addressed first.

| Sta